In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
df = pd.read_csv(
    "C:/Users/sadaa/Downloads/Datamites/client project/project_purchase_pattern_analysis.csv",
    low_memory=False
)

In [5]:
df.head(4)

,BillNo,Itemname,Quantity,Present_Date,Price,CustomerID,Country
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6,01-12-2010 08:26,2.55,17850.0,United Kingdom
1,536365,WHITE METAL LANTERN,6,01-12-2010 08:26,3.39,17850.0,United Kingdom
2,536365,CREAM CUPID HEARTS COAT HANGER,8,01-12-2010 08:26,2.75,17850.0,United Kingdom
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6,01-12-2010 08:26,3.39,17850.0,United Kingdom


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 522064 entries, 0 to 522063
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   BillNo        522064 non-null  object 
 1   Itemname      520609 non-null  object 
 2   Quantity      522064 non-null  int64  
 3   Present_Date  522064 non-null  object 
 4   Price         522064 non-null  float64
 5   CustomerID    388023 non-null  float64
 6   Country       522064 non-null  object 
dtypes: float64(2), int64(1), object(4)
memory usage: 27.9+ MB


In [9]:
df.describe()

,Quantity,Price,CustomerID
count,522064.000000,522064.000000,388023.000000
mean,10.090435,3.826801,15316.931710
std,161.110525,41.900599,1721.846964
min,-9600.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13950.000000
50%,3.000000,2.080000,15265.000000
75%,10.000000,4.130000,16837.000000
max,80995.000000,13541.330000,18287.000000


In [11]:
df.isnull().sum()

BillNo               0
Itemname          1455
Quantity             0
Present_Date         0
Price                0
CustomerID      134041
Country              0
dtype: int64

In [14]:
df.isnull().sum()/df.shape[0]*100

BillNo           0.000000
Itemname         0.278701
Quantity         0.000000
Present_Date     0.000000
Price            0.000000
CustomerID      25.675205
Country          0.000000
dtype: float64

In [16]:
df["Present_Date"] = pd.to_datetime(df["Present_Date"], dayfirst=True, errors="coerce")
print(df["Present_Date"].dtype)

datetime64[ns]


In [18]:
df["CustomerID"] = df["CustomerID"].fillna("Guest")
df["CustomerID"] = df["CustomerID"].astype(str)
df["CustomerID"] = df["CustomerID"].str.replace(".0", "", regex=False)
print(df["CustomerID"].dtype)

object


In [20]:
#Reomve the missing itemname 
df = df.dropna(subset=["Itemname"])
df.shape

(520609, 7)

In [22]:
# Rows where Quantity <= 0
invalid_qty = df[df["Quantity"] <= 0]
print("Invalid Quantity rows:", len(invalid_qty))

# Rows where Price <= 0
invalid_price = df[df["Price"] <= 0]
print("Invalid Price rows:", len(invalid_price))

Invalid Quantity rows: 473
Invalid Price rows: 1058


In [24]:
#Filters the dataset
#It keeps only valid transactions where both quantity and price are greater than zero.
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

In [26]:
print("Remaining rows:", len(df))
print("Quantity <= 0 remaining:", (df["Quantity"] <= 0).sum())
print("Price <= 0 remaining:", (df["Price"] <= 0).sum())

Remaining rows: 519551
Quantity <= 0 remaining: 0
Price <= 0 remaining: 0


In [30]:
df.isnull().sum()

BillNo          0
Itemname        0
Quantity        0
Present_Date    0
Price           0
CustomerID      0
Country         0
dtype: int64

In [28]:
df.describe()

,Quantity,Present_Date,Price
count,519551.000000,519551,519551.000000
mean,10.398361,2011-07-04 16:03:31.051080704,3.887894
min,1.000000,2010-12-01 08:26:00,0.001000
25%,1.000000,2011-03-28 10:52:00,1.250000
50%,3.000000,2011-07-20 11:55:00,2.080000
75%,10.000000,2011-10-19 15:08:00,4.130000
max,80995.000000,2011-12-09 12:50:00,13541.330000
std,157.004952,NaN,35.954045


In [32]:
df.to_csv("Purchase_Pattern_cleaned_data.csv", index=False)

In [34]:
#Group by BillNo and aggregate Itemname into lists
basket = df.groupby("BillNo")["Itemname"].apply(list)
basket.head(5)

BillNo
536365    [WHITE HANGING HEART T-LIGHT HOLDER, WHITE MET...
536366    [HAND WARMER UNION JACK, HAND WARMER RED POLKA...
536367    [ASSORTED COLOUR BIRD ORNAMENT, POPPY'S PLAYHO...
536368    [JAM MAKING SET WITH JARS, RED COAT RACK PARIS...
536369                           [BATH BUILDING BLOCK WORD]
Name: Itemname, dtype: object

In [36]:
basket.shape

(19559,)

In [38]:
df['Itemname'].nunique()

4006

In [40]:
te = TransactionEncoder()
te_ary = te.fit(basket).transform(basket)
basket_df = pd.DataFrame(te_ary, columns=te.columns_)
basket_df

,*Boombox Ipod Classic,*USB Office Mirror Ball,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 DAISY PEGS IN WOOD BOX,12 EGG HOUSE PAINTED WOOD,12 HANGING EGGS HAND PAINTED,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,...,ZINC STAR T-LIGHT HOLDER,ZINC SWEETHEART SOAP DISH,ZINC SWEETHEART WIRE LETTER RACK,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC TOP 2 DOOR WOODEN SHELF,ZINC WILLIE WINKIE CANDLE STICK,ZINC WIRE KITCHEN ORGANISER,ZINC WIRE SWEETHEART LETTER TRAY
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19554,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19555,False,False,False,False,False,False,False,False,False,False,...,False,False,False,True,False,False,False,True,False,False
19556,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
19557,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [42]:
basket_df.shape

(19559, 4006)

In [44]:
frequent_itemsets = apriori(basket_df, min_support=0.02, use_colnames=True)
frequent_itemsets

,support,itemsets
0,0.023365,(3 STRIPEY MICE FELTCRAFT)
1,0.023672,(4 TRADITIONAL SPINNING TOPS)
2,0.048162,(6 RIBBONS RUSTIC CHARM)
3,0.021320,(60 CAKE CASES DOLLY GIRL DESIGN)
4,0.030676,(60 CAKE CASES VINTAGE CHRISTMAS)
...,...,...
377,0.020655,"(STRAWBERRY CHARLOTTE BAG, WOODLAND CHARLOTTE ..."
378,0.020707,"(WHITE HANGING HEART T-LIGHT HOLDER, WOODEN PI..."
379,0.027404,"(WOODEN FRAME ANTIQUE WHITE, WOODEN PICTURE FR..."
380,0.026279,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC..."


In [45]:
itemsets_with_length = frequent_itemsets.copy()
itemsets_with_length["length"] = itemsets_with_length["itemsets"].apply(lambda x: len(x))
itemsets_with_length

,support,itemsets,length
0,0.023365,(3 STRIPEY MICE FELTCRAFT),1
1,0.023672,(4 TRADITIONAL SPINNING TOPS),1
2,0.048162,(6 RIBBONS RUSTIC CHARM),1
3,0.021320,(60 CAKE CASES DOLLY GIRL DESIGN),1
4,0.030676,(60 CAKE CASES VINTAGE CHRISTMAS),1
...,...,...,...
377,0.020655,"(STRAWBERRY CHARLOTTE BAG, WOODLAND CHARLOTTE ...",2
378,0.020707,"(WHITE HANGING HEART T-LIGHT HOLDER, WOODEN PI...",2
379,0.027404,"(WOODEN FRAME ANTIQUE WHITE, WOODEN PICTURE FR...",2
380,0.026279,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",3


In [46]:
frequent_itemset_top_20 = frequent_itemsets.sort_values(by="support", ascending=False).head(20)
frequent_itemset_top_20 = frequent_itemset_top_20.reset_index(drop=True)
frequent_itemset_top_20

,support,itemsets
0,0.112378,(WHITE HANGING HEART T-LIGHT HOLDER)
1,0.105373,(JUMBO BAG RED RETROSPOT)
2,0.097346,(REGENCY CAKESTAND 3 TIER)
3,0.084616,(PARTY BUNTING)
4,0.078787,(LUNCH BAG RED RETROSPOT)
5,0.073163,(ASSORTED COLOUR BIRD ORNAMENT)
6,0.068817,(SET OF 3 CAKE TINS PANTRY DESIGN)
7,0.065392,(PACK OF 72 RETROSPOT CAKE CASES)
8,0.064420,(LUNCH BAG BLACK SKULL.)
9,0.062989,(NATURAL SLATE HEART CHALKBOARD)


In [47]:
top_pairs = itemsets_with_length[itemsets_with_length["length"] == 2].sort_values("support", ascending=False)
top_pairs

,support,itemsets,length
324,0.041873,"(JUMBO BAG PINK POLKADOT, JUMBO BAG RED RETROS...",2
314,0.037323,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",2
337,0.036863,"(JUMBO STORAGE BAG SUKI, JUMBO BAG RED RETROSPOT)",2
335,0.034562,"(JUMBO SHOPPER VINTAGE RED PAISLEY, JUMBO BAG ...",2
300,0.032364,"(ALARM CLOCK BAKELIKE RED, ALARM CLOCK BAKELIK...",2
...,...,...,...
376,0.020553,"(SET OF 3 CAKE TINS PANTRY DESIGN, SET OF 6 SP...",2
322,0.020502,"(JUMBO BAG PEARS, JUMBO BAG APPLES)",2
310,0.020195,"(JUMBO STORAGE BAG SUKI, DOTCOM POSTAGE)",2
361,0.020093,"(LUNCH BAG WOODLAND, LUNCH BAG PINK POLKADOT)",2


In [48]:
top_triples = itemsets_with_length[itemsets_with_length["length"] == 3].sort_values("support", ascending=False)
top_triples

,support,itemsets,length
380,0.026279,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",3
381,0.021064,"(JUMBO STORAGE BAG SUKI, JUMBO BAG PINK POLKAD...",3


In [54]:
raw_rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)
rules = raw_rules[['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift']].sort_values(by="lift", ascending =False)
rules

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
164,"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.037323,0.037579,0.026279,0.704110,18.736979
169,(PINK REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",0.037579,0.037323,0.026279,0.699320,18.736979
165,"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.029091,0.049747,0.026279,0.903339,18.158696
168,(GREEN REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",0.049747,0.029091,0.026279,0.528263,18.158696
28,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.049747,0.037579,0.030881,0.620761,16.518987
...,...,...,...,...,...,...,...
161,(WOODEN PICTURE FRAME WHITE FINISH),(WHITE HANGING HEART T-LIGHT HOLDER),0.055831,0.112378,0.020707,0.370879,3.300284
138,(WHITE HANGING HEART T-LIGHT HOLDER),(NATURAL SLATE HEART CHALKBOARD),0.112378,0.062989,0.021013,0.186988,2.968589
139,(NATURAL SLATE HEART CHALKBOARD),(WHITE HANGING HEART T-LIGHT HOLDER),0.062989,0.112378,0.021013,0.333604,2.968589
87,(JUMBO BAG RED RETROSPOT),(WHITE HANGING HEART T-LIGHT HOLDER),0.105373,0.112378,0.022445,0.213003,1.895420


In [56]:
rules['rule'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x))) + " --> " + rules['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules = rules[['rule'] + [col for col in rules.columns if col != 'rule']]
rules = rules.reset_index(drop=True)
rules

,rule,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...","(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.037323,0.037579,0.026279,0.704110,18.736979
1,PINK REGENCY TEACUP AND SAUCER --> GREEN REGEN...,(PINK REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",0.037579,0.037323,0.026279,0.699320,18.736979
2,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...","(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.029091,0.049747,0.026279,0.903339,18.158696
3,GREEN REGENCY TEACUP AND SAUCER --> PINK REGEN...,(GREEN REGENCY TEACUP AND SAUCER),"(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",0.049747,0.029091,0.026279,0.528263,18.158696
4,GREEN REGENCY TEACUP AND SAUCER --> PINK REGEN...,(GREEN REGENCY TEACUP AND SAUCER),(PINK REGENCY TEACUP AND SAUCER),0.049747,0.037579,0.030881,0.620761,16.518987
...,...,...,...,...,...,...,...,...
171,WOODEN PICTURE FRAME WHITE FINISH --> WHITE HA...,(WOODEN PICTURE FRAME WHITE FINISH),(WHITE HANGING HEART T-LIGHT HOLDER),0.055831,0.112378,0.020707,0.370879,3.300284
172,WHITE HANGING HEART T-LIGHT HOLDER --> NATURAL...,(WHITE HANGING HEART T-LIGHT HOLDER),(NATURAL SLATE HEART CHALKBOARD),0.112378,0.062989,0.021013,0.186988,2.968589
173,NATURAL SLATE HEART CHALKBOARD --> WHITE HANGI...,(NATURAL SLATE HEART CHALKBOARD),(WHITE HANGING HEART T-LIGHT HOLDER),0.062989,0.112378,0.021013,0.333604,2.968589
174,JUMBO BAG RED RETROSPOT --> WHITE HANGING HEAR...,(JUMBO BAG RED RETROSPOT),(WHITE HANGING HEART T-LIGHT HOLDER),0.105373,0.112378,0.022445,0.213003,1.895420


In [58]:
Top_rules = rules[(rules['lift'] > 5) & (rules['confidence'] > 0.7)]
Top_rules

,rule,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
0,"GREEN REGENCY TEACUP AND SAUCER, ROSES REGENCY...","(ROSES REGENCY TEACUP AND SAUCER, GREEN REGENC...",(PINK REGENCY TEACUP AND SAUCER),0.037323,0.037579,0.026279,0.704110,18.736979
2,"PINK REGENCY TEACUP AND SAUCER, ROSES REGENCY ...","(ROSES REGENCY TEACUP AND SAUCER, PINK REGENCY...",(GREEN REGENCY TEACUP AND SAUCER),0.029091,0.049747,0.026279,0.903339,18.158696
5,PINK REGENCY TEACUP AND SAUCER --> GREEN REGEN...,(PINK REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.037579,0.049747,0.030881,0.821769,16.518987
6,"GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY ...","(GREEN REGENCY TEACUP AND SAUCER, PINK REGENCY...",(ROSES REGENCY TEACUP AND SAUCER),0.030881,0.051741,0.026279,0.850993,16.447213
8,GARDENERS KNEELING PAD CUP OF TEA --> GARDENER...,(GARDENERS KNEELING PAD CUP OF TEA),(GARDENERS KNEELING PAD KEEP CALM),0.038550,0.046270,0.027813,0.721485,15.592854
10,PINK REGENCY TEACUP AND SAUCER --> ROSES REGEN...,(PINK REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.037579,0.051741,0.029091,0.774150,14.962049
12,GREEN REGENCY TEACUP AND SAUCER --> ROSES REGE...,(GREEN REGENCY TEACUP AND SAUCER),(ROSES REGENCY TEACUP AND SAUCER),0.049747,0.051741,0.037323,0.750257,14.500272
13,ROSES REGENCY TEACUP AND SAUCER --> GREEN REGE...,(ROSES REGENCY TEACUP AND SAUCER),(GREEN REGENCY TEACUP AND SAUCER),0.051741,0.049747,0.037323,0.721344,14.500272
19,CHARLOTTE BAG PINK POLKADOT --> RED RETROSPOT ...,(CHARLOTTE BAG PINK POLKADOT),(RED RETROSPOT CHARLOTTE BAG),0.037681,0.052457,0.026586,0.705563,13.450398
73,"JUMBO BAG PINK POLKADOT, JUMBO STORAGE BAG SUK...","(JUMBO STORAGE BAG SUKI, JUMBO BAG PINK POLKADOT)",(JUMBO BAG RED RETROSPOT),0.026279,0.105373,0.021064,0.801556,7.606813


In [60]:
# --- Convert itemsets to string ---
itemsets_with_length["itemsets"] = itemsets_with_length["itemsets"].apply(
    lambda x: ", ".join(sorted(list(x)))
)

In [62]:
itemsets_with_length.to_csv("Frequent_itemsets.csv", index=False)

In [64]:
# convert frozenset -> sorted comma string, replacing the original columns
rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(list(x))))
rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(list(x))))

# create readable rule text
rules["rule"] = rules["antecedents"] + " -> " + rules["consequents"]

rules.to_csv("Association_rules.csv", index=False)